# PyMAUDE — Example Notebook

**PyMAUDE** is a Python library for accessing and analyzing FDA MAUDE (Manufacturer and User Facility Device Experience) adverse event data via a fast [DuckDB](https://duckdb.org/) backend.

This notebook downloads real FDA MAUDE data and demonstrates all major library capabilities using venous stent and rotational thrombectomy devices as worked examples.

> **First run:** `add_years(..., download=True)` downloads zip files from the FDA FTP server (~few hundred MB for 3 years of device/master/text). Subsequent runs use the local cache and skip unchanged files via SHA-256 checksum.

---
## Contents
1. [Setup](#1-setup)
2. [Download & load data](#2-load)
3. [Inspect the database](#3-inspect)
4. [Exact-field queries](#4-exact)
5. [Substring search](#5-search)
6. [Grouped search across device classes](#6-grouped)
7. [Event narratives](#7-narratives)
8. [Enrich with patient outcomes](#8-patient)
9. [Enrich with device problem codes](#9-problems)
10. [Trend analysis by year](#10-trends)
11. [Chained filters](#11-filters)
12. [Raw SQL](#12-sql)
13. [Archive a snapshot for publication](#13-archive)

---
## 1. Setup <a id="1-setup"></a>

In [ ]:
from pymaude import MaudeDatabase

DB_PATH  = './maude.duckdb'   # persistent DuckDB file
DATA_DIR = './maude_data'     # downloaded zip/txt files live here
YEARS    = '2024-2026'        # adjust to taste; more years = more data

---
## 2. Download & load data <a id="2-load"></a>

Pass `download=True` to fetch from the FDA FTP area. Files are cached locally — re-running this cell only downloads files whose SHA-256 checksum has changed since the last run.

In [ ]:
db = MaudeDatabase(DB_PATH, data_dir=DATA_DIR, verbose=True, memory_limit='2GB')
db.info()

db.add_years(
    YEARS,
    tables=['master', 'device', 'text', 'patient', 'problem'],
    # download=True
    download=False
)

# db.update()

To pull in the latest monthly FDA updates at any point — this is also the safest way to refresh cumulative tables (`master`, `patient`, `problem`), since it automatically re-requests every year they already have loaded:
```python
db.update(download=True)
```

Call `add_years()` again at any time to extend coverage or add more tables — here we add `device`, `text`, and `patient` outcomes alongside the same year range.

Note: `master` and `problem` are cumulative tables — they're fully replaced on every reload rather than updated year-by-year, so `add_years()` rejects a request that doesn't cover every year they already have loaded (it would otherwise silently drop the missing years). That's why this passes the full `YEARS` range again rather than just `'2026'` — pass `force_partial=True` if you really do want to narrow an existing cumulative table's coverage.

---
## 3. Inspect the database <a id="3-inspect"></a>

In [ ]:
db.info()

---
## 4. Exact-field queries <a id="4-exact"></a>

`query_device()` does **exact** matching: case-insensitive for `brand_name`, `generic_name`, and `manufacturer_name`, but **case-sensitive** for `product_code` (FDA product codes are always uppercase, e.g. `'NIQ'` — `'niq'` won't match). Combine any of these plus `start_date`/`end_date`.

In [ ]:
# All events for a specific product code (NIQ = venous stents)
niq = db.query_device(product_code='NIQ')
print(f'NIQ (venous stent) events loaded: {len(niq):,}')
niq[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'MANUFACTURER_D_NAME', 'DATE_RECEIVED']].head(10)

In [ ]:
# Narrow to a specific brand + date window
bsci_jan_thru_aug_2021 = db.query_device(
    brand_name='Boston Scientific',
    start_date='2021-01-01',
    end_date='2021-08-31'
)

print(f'Boston Scientific events in 2021 Jan-Aug: {len(bsci_jan_thru_aug_2021):,}')
bsci_jan_thru_aug_2021[['MDR_REPORT_KEY', 'BRAND_NAME', 'EVENT_TYPE', 'MANUFACTURER_G1_NAME', 'DATE_RECEIVED']]

---
## 5. Substring search <a id="5-search"></a>

`search_by_device_names()` does **case-insensitive substring matching** across a synthesized `DEVICE_NAME_CONCAT` column (BRAND_NAME | GENERIC_NAME | MANUFACTURER_D_NAME concatenated). This is useful because MAUDE entries are often inconsistent, with names of medical devices often appearing in one of these three columns. 

| Criteria format | Logic |
|---|---|
| `'term'` | single substring |
| `['a', 'b']` | a **OR** b |
| `[['a', 'b'], 'c']` | (a **AND** b) **OR** c |
| `{'group1': ..., 'group2': ...}` | grouped (see [section 6](#6-grouped)) |

In [ ]:
# Simple substring — anything with "venous stent" in any name field
venous_stents = db.search_by_device_names('venous stent')
print(f'Events matching "venous stent": {len(venous_stents):,}')
venous_stents[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'DATE_RECEIVED']]

In [ ]:
# OR logic — venous stent OR iliac stent
venous_iliac = db.search_by_device_names(['venous stent', 'iliac stent'])
print(f'Venous OR iliac stents: {len(venous_iliac):,}')
venous_iliac['GENERIC_NAME'].value_counts().head(10)

In [ ]:
# AND logic — must contain both "argon" AND "cleaner" (avoids false positives)
argon_cleaner = db.search_by_device_names([['argon','cleaner']])
print(f'Argon AND Cleaner events: {len(argon_cleaner):,}')
argon_cleaner[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'MANUFACTURER_D_NAME', 'DATE_RECEIVED']]

In [ ]:
# Combined: (argon AND cleaner) OR (angiojet) OR (thrombectomy)
thrombectomy_broad = db.search_by_device_names(
    [['argon', 'cleaner'], 'angiojet', 'rotational thrombectomy']
)
print(f'Broad rotational thrombectomy search: {len(thrombectomy_broad):,}')
thrombectomy_broad[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME']].head(10)

Results are pandas DataFrames — export anytime:

In [ ]:
thrombectomy_broad.to_csv('thrombectomy_broad.csv', index=False)

---
## 6. Grouped search across device classes <a id="6-grouped"></a>

A **dict** of criteria runs multiple searches at once and labels each result with a `search_group` column. Useful for comparative studies.

In [ ]:
grouped = db.search_by_device_names({
    'venous_stents':  ['venous stent', 'venous stenting'],
    'thrombectomy':   [['argon', 'cleaner'], 'angiojet', 'rotational thrombectomy'],
})

print(f'Total events: {len(grouped):,}')
print('\nEvents by group:')
print(grouped['search_group'].value_counts().to_string())

In [ ]:
grouped[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME', 'search_group']].head(15)

---
## 7. Event narratives <a id="7-narratives"></a>

`get_narratives()` fetches the free-text FOI_TEXT descriptions for a set of MDR_REPORT_KEYs.

In [ ]:
narratives = db.get_narratives(thrombectomy_broad['MDR_REPORT_KEY'])
print(f'Narratives retrieved: {len(narratives):,}')
narratives.head(5)

In [ ]:
# Read a few narratives in full
for _, row in narratives.head(3).iterrows():
    print(f"=== MDR {row['MDR_REPORT_KEY']} ===")
    print(row['FOI_TEXT'][:500])
    print()

---
## 8. Enrich with patient outcomes <a id="8-patient"></a>

`enrich_with_patient_data()` left-joins the patient outcomes table. `SEQUENCE_NUMBER_OUTCOME` codes:
- `D` Death · `L` Life Threatening · `H` Hospitalization · `S` Disability · `C` Congenital Anomaly · `R` Required Intervention · `O` Other · `U` Unknown · `I` No Information · `A` Not Applicable · `*` Invalid Data

A single patient record can carry more than one code at once, e.g. `"H; O"` — so don't match this field with `==`.

In [ ]:
enriched = db.enrich_with_patient_data(thrombectomy_broad)

with_outcome = enriched.dropna(subset=['SEQUENCE_NUMBER_OUTCOME'])
print(f'Events with patient outcome data: {len(with_outcome):,} / {len(enriched):,}')
with_outcome[['MDR_REPORT_KEY', 'BRAND_NAME', 'SEQUENCE_NUMBER_OUTCOME']].head(10)

In [ ]:
OUTCOME_LABELS = {
    'D': 'Death', 'L': 'Life Threatening', 'H': 'Hospitalization',
    'S': 'Disability', 'C': 'Congenital Anomaly', 'R': 'Required Intervention',
    'O': 'Other', 'U': 'Unknown', 'I': 'No Information', 'A': 'Not Applicable',
    '*': 'Invalid Data',
}

outcome_counts = (
    with_outcome['SEQUENCE_NUMBER_OUTCOME']
    .str.split(';')
    .explode()
    .str.strip()
    .map(lambda c: OUTCOME_LABELS.get(c, c))
    .value_counts()
)
outcome_counts

---
## 9. Enrich with device problem codes <a id="9-problems"></a>

`enrich_with_problems()` left-joins the device problem code table (`problem`, loaded from 2000 onwards per `TABLE_METADATA`).

In [ ]:
with_problems = db.enrich_with_problems(thrombectomy_broad)

problem_counts = (
    with_problems
    .dropna(subset=['DEVICE_PROBLEM_CODE'])
    ['DEVICE_PROBLEM_CODE']
    .value_counts()
)
print(f'Events with a problem code: {problem_counts.sum():,}')
print('\nTop problem codes:')
problem_counts.head(10)

---
## 10. Trend analysis by year <a id="10-trends"></a>

`get_trends_by_year()` counts events per calendar year. When results include a `search_group` column it breaks down by group automatically.

In [ ]:
# Overall year-over-year trend
trends = db.get_trends_by_year(thrombectomy_broad)
print('Rotational thrombectomy — events by year:')
print(trends.to_string(index=False))

In [ ]:
# Per-group trends
group_trends = db.get_trends_by_year(grouped)
print('Events by year and device group:')
print(group_trends.to_string(index=False))

In [ ]:
# Pivot for side-by-side comparison
if 'search_group' in group_trends.columns:
    pivot = (
        group_trends
        .pivot(index='year', columns='search_group', values='event_count')
        .fillna(0).astype(int)
    )
    print(pivot.to_string())

---
## 11. Chained filters <a id="11-filters"></a>

The `filter_by_*` methods take a results DataFrame and return a narrowed-down copy — there's no query-builder object, just plain DataFrames passed from one call to the next.

| Method | Requires first | Matches on |
|---|---|---|
| `filter_by_outcome(df, code)` | `enrich_with_patient_data()` | `SEQUENCE_NUMBER_OUTCOME` — membership, since one record can carry multiple codes (e.g. `"H; O"`) |
| `filter_by_patient(df, age_min=, age_max=, sex=)` | `enrich_with_patient_data()` | `PATIENT_AGE` / `PATIENT_SEX` |
| `filter_by_problem(df, code)` | `enrich_with_problems()` | `DEVICE_PROBLEM_CODE` |
| `filter_by_narrative(df, term)` | nothing — queries `text` directly | `FOI_TEXT` substring |

Each raises a clear `ValueError` if the required enrich step hasn't been run yet.

In [ ]:
# Deaths among the thrombectomy events (reusing `enriched` from section 8)
deaths = db.filter_by_outcome(enriched, 'D')
print(f'Deaths: {len(deaths):,} / {len(enriched):,} enriched records')
deaths[['MDR_REPORT_KEY', 'BRAND_NAME', 'SEQUENCE_NUMBER_OUTCOME']].head(10)

In [ ]:
# Demographic filters — age range and/or sex
elderly = db.filter_by_patient(enriched, age_min=65)
print(f'Events in patients 65+: {len(elderly):,}')

females = db.filter_by_patient(enriched, sex='Female')
print(f'Events in female patients: {len(females):,}')

In [ ]:
# Filter to a specific problem code (reusing `with_problems`/`problem_counts` from section 9)
top_code = problem_counts.index[0]
events_with_top_code = db.filter_by_problem(with_problems, top_code)
print(f'Events with problem code {top_code!r}: {len(events_with_top_code):,}')
events_with_top_code[['MDR_REPORT_KEY', 'BRAND_NAME', 'DEVICE_PROBLEM_CODE']].head(10)

In [ ]:
# Narrative keyword filter — no enrich step required, queries `text` directly
battery_related = db.filter_by_narrative(thrombectomy_broad, 'battery')
print(f'Narratives mentioning "battery": {len(battery_related):,}')
battery_related[['MDR_REPORT_KEY', 'BRAND_NAME', 'GENERIC_NAME']].head(10)

In [ ]:
# Chaining several filters: device search → enrich → outcome → demographic
step1 = db.enrich_with_patient_data(thrombectomy_broad)
step2 = db.filter_by_outcome(step1, 'D')
fatal_elderly = db.filter_by_patient(step2, age_min=65)

print(f'Fatal thrombectomy events in patients 65+: {len(fatal_elderly):,}')
fatal_elderly[['MDR_REPORT_KEY', 'BRAND_NAME', 'PATIENT_AGE', 'SEQUENCE_NUMBER_OUTCOME']]

---
## 12. Raw SQL <a id="12-sql"></a>

`db.query()` exposes DuckDB directly. Tables available: `master`, `device`, `text`, `patient`, `problem`.

In [ ]:
# Event type breakdown across all loaded data
db.query("""
    SELECT
        EVENT_TYPE,
        CASE EVENT_TYPE
            WHEN 'D'  THEN 'Death'
            WHEN 'IN' THEN 'Injury'
            WHEN 'M'  THEN 'Malfunction'
            WHEN 'O'  THEN 'Other'
            ELSE EVENT_TYPE
        END AS label,
        COUNT(*) AS n
    FROM master
    GROUP BY EVENT_TYPE
    ORDER BY n DESC
""")

In [ ]:
# Top 10 manufacturers by adverse event volume (master joined to device)
db.query("""
    SELECT
        d.MANUFACTURER_D_NAME,
        COUNT(DISTINCT m.MDR_REPORT_KEY) AS events
    FROM master m
    JOIN device d USING (MDR_REPORT_KEY)
    GROUP BY d.MANUFACTURER_D_NAME
    ORDER BY events DESC
    LIMIT 10
""")

In [ ]:
# Parameterized query — safe for user-supplied input
db.query(
    "SELECT MDR_REPORT_KEY, BRAND_NAME, DATE_RECEIVED FROM device WHERE DEVICE_REPORT_PRODUCT_CODE = ? LIMIT 10",
    params=['NIQ']
)

---
## 13. Archive a snapshot for publication <a id="13-archive"></a>

MAUDE is updated continuously and isn't versioned, so a query run today may return different results in a year. `db.archive()` freezes the exact database backing an analysis so it can be cited or uploaded alongside a paper (e.g. to Zenodo):

- Checkpoints and copies the DuckDB file itself.
- Writes a `manifest.json` recording, per loaded table/year: source file, SHA-256 checksum, row count, and load timestamp — plus the DuckDB and pymaude versions used to build it.
- With `include_raw=True`, also copies the raw MAUDE source files referenced in the manifest into `output_dir/raw/`, so reviewers can re-derive the database from scratch.

In [ ]:
manifest_path = db.archive('./maude_archive', include_raw=False)
print(f'Archive written, manifest at: {manifest_path}')

In [ ]:
# Inspect the generated manifest — this is what accompanies the archived .duckdb file
import json

with open(manifest_path) as f:
    manifest = json.load(f)

print(f"pymaude {manifest['pymaude_version']}  ·  duckdb {manifest['duckdb_version']}  ·  {manifest['checksum_algorithm']}")
print(f"Database: {manifest['database']['filename']} ({manifest['database']['size_bytes']:,} bytes)")
print(f"Tables tracked: {len(manifest['tables'])}")
manifest['tables'][:5]

In [ ]:
db.close()

